# 06 · Time Series: Tracking Trends Over Time

Build multi-month time series from country-level Monthly Stats.

## Why trends matter more than snapshots

A single month's ranking tells you who is ahead *right now*, but not whether things
are improving or degrading. Time series reveal:

- **Seasonal patterns** — some countries show higher latency in summer (more users)
- **Infrastructure rollouts** — a sudden jump in median download marks a major upgrade
- **COVID-19 effects** — 2020 data shows dramatic shifts in latency and loss as usage
  patterns changed overnight
- **Long-run convergence** — developing-country speeds have risen faster than
  developed-country speeds since 2015

## Loading note

Each month is a separate parquet file. The disk cache means you only download each
file once — `./cache/v1/…` stores them for reuse across sessions.

## Setup

In [7]:
import json
from pathlib import Path

import pandas as pd
import requests

try:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import seaborn as sns
    from IPython.display import clear_output, display
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 5)
except ImportError as e:
    print(f"Note: {e}")
    print("  Install with: uv add matplotlib seaborn ipywidgets")


In [8]:
# ── Country name lookup ──────────────────────────────────────────────────────
# countrylookup.py is a local helper (same directory as this notebook) that
# converts ISO 3166-1 alpha-2 codes to readable English country names.
# It tries pycountry → restcountries.com API → built-in fallback dict.
#
# If you move this notebook, keep countrylookup.py alongside it.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # ensure local module is found
from countrylookup import cc_name, cc_label

print(f"Country lookup ready — {cc_label('US')}, {cc_label('KR')}, {cc_label('XK')}")

Country lookup ready — United States (US), South Korea (KR), Kosovo (XK)


In [9]:
# ── Discover available months ─────────────────────────────────────────────────
#
# Fetch the M-Lab manifest to learn which months are available per slice.
# Each entry includes the public download URL and the local cache path.

MANIFEST_URL = "https://measurementlab.net/data/iqb/manifest.json"
resp = requests.get(MANIFEST_URL, timeout=30)
resp.raise_for_status()

records = []
for path, meta in resp.json()["files"].items():
    parts = path.split("/")
    if len(parts) >= 6 and parts[5] == "data.parquet":
        records.append({
            "start":      pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "slice":      parts[4],
            "url":        meta["url"],
            "cache_path": path,
        })

catalog = (
    pd.DataFrame(records)
    .sort_values(["slice", "start"])
    .reset_index(drop=True)
)

print(f"Catalog: {len(catalog)} entries, {catalog['slice'].nunique()} slices, "
      f"{catalog['start'].min().date()} → {catalog['start'].max().date()}")


Catalog: 2450 entries, 12 slices, 2009-01-01 → 2026-01-01


In [10]:
# ── Data loader ──────────────────────────────────────────────────────────────
#
# Downloads parquet files from the public M-Lab URLs in the manifest.
# Files are cached to ./cache/v1/... on first access and reused on subsequent
# runs (matching the path structure used by the iqb library's local cache).

from io import BytesIO

_mem_cache: dict = {}

def load_parquet(slice_name: str, start: str) -> pd.DataFrame:
    key = (slice_name, start)
    if key in _mem_cache:
        return _mem_cache[key]

    start_ts = pd.to_datetime(start)
    row = catalog[(catalog["slice"] == slice_name) & (catalog["start"] == start_ts)]
    if row.empty:
        available = catalog[catalog["slice"] == slice_name]["start"].dt.strftime("%Y-%m-%d").tolist()
        raise ValueError(f"No data for slice='{slice_name}', month='{start}'.\nAvailable: {available}")
    row = row.iloc[0]

    local_path = Path(row["cache_path"])
    if local_path.exists():
        df = pd.read_parquet(local_path)
    else:
        print(f"[download] {slice_name} / {start} …")
        r = requests.get(row["url"], timeout=60)
        r.raise_for_status()
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(r.content)
        df = pd.read_parquet(BytesIO(r.content))
        print(f"  ✓ saved to {local_path}  ({len(df):,} rows)")

    _mem_cache[key] = df
    return df


## Interactive Four-Panel Time Series

Select countries (Ctrl/Cmd+click for multiple) and a month range, then click
**Load**. All four metrics appear side by side.

In [11]:
ts_months = sorted(
    catalog[catalog["slice"] == "downloads_by_country"]["start"]
    .dt.strftime("%Y-%m-%d").unique(), reverse=True,
)

_latest = load_parquet("downloads_by_country", ts_months[0])
all_countries = sorted(_latest["country_code"].dropna().unique())

w_countries = widgets.SelectMultiple(
    options=[(cc_label(c), c) for c in all_countries], value=["US","DE","BR","IN"],
    description="Countries:", rows=10, layout=widgets.Layout(width="200px"),
)
w_n = widgets.IntSlider(value=12, min=3, max=min(24,len(ts_months)), step=1,
                         description="Months:", layout=widgets.Layout(width="380px"))
w_run = widgets.Button(description="Load / Refresh", button_style="primary",
                        layout=widgets.Layout(width="150px"))
out   = widgets.Output()

def run(b=None):
    selected = list(w_countries.value)
    months_to_use = list(reversed(ts_months[:w_n.value]))  # chronological order
    with out:
        clear_output(wait=True)
        print(f"Loading {len(months_to_use)} months for {selected} ...")
        records = []
        for m in months_to_use:
            try:
                dl = load_parquet("downloads_by_country", m)
                ul = load_parquet("uploads_by_country", m)
                ul_cols = [c for c in ul.columns if c.startswith("upload_")]
                df = dl.merge(ul[["country_code"] + ul_cols], on="country_code", how="left")
                sel = df[df["country_code"].isin(selected)].copy()
                sel["month"] = pd.to_datetime(m)
                records.append(sel)
            except Exception as e:
                print(f"  Skipping {m}: {e}")
        if not records: print("No data loaded."); return
        ts = pd.concat(records, ignore_index=True)

        fig, axes = plt.subplots(2,2,figsize=(14,9))
        panels = [
            ("download_p50","Median download (Mbit/s)",  axes[0,0]),
            ("upload_p50",  "Median upload (Mbit/s)",    axes[0,1]),
            ("latency_p50", "Median latency (ms) - lower is better",  axes[1,0]),
            ("loss_p50",    "Median packet loss - lower is better",   axes[1,1]),
        ]
        for col, ylabel, ax in panels:
            for cc, grp in ts.groupby("country_code"):
                grp = grp.sort_values("month")
                ax.plot(grp["month"],grp[col],marker="o",markersize=4,label=cc_label(cc))
            ax.set_ylabel(ylabel)
            ax.xaxis.set_major_locator(mticker.MaxNLocator(6))
            plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha="right")
            ax.legend(fontsize=8)
        plt.suptitle(
            f"IQB country trends — {ts['month'].min().date()} to {ts['month'].max().date()}",
            fontsize=13)
        plt.tight_layout(); plt.show()

w_run.on_click(run)
display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([widgets.Label("Countries (Ctrl+click to multi-select):"),w_countries]),
        widgets.VBox([w_n,w_run]),
    ]),
    out,
]))
run()

## Percentile Band Over Time

The median alone hides *distribution shifts*. If the median and p75 both rise but
p25 stays flat, the top half of connections improved while the bottom half did not.

The shaded band spans **p25–p75** (the middle 50% of connections). A narrowing
band means the gap between fast and slow users is shrinking.

In [12]:
w_bc  = widgets.Dropdown(options=[(cc_label(c), c) for c in all_countries], value="US", description="Country:",
                          layout=widgets.Layout(width="200px"))
w_bme = widgets.Dropdown(
    options=[("Download (Mbit/s)","download"),("Upload (Mbit/s)","upload"),
             ("Latency (ms)","latency"),("Packet Loss","loss")],
    description="Metric:", layout=widgets.Layout(width="260px"),
)
w_bn  = widgets.IntSlider(value=12, min=3, max=min(24,len(ts_months)), step=1,
                           description="Months:", layout=widgets.Layout(width="380px"))
w_brun = widgets.Button(description="Load / Refresh", button_style="primary",
                         layout=widgets.Layout(width="150px"))
out_b  = widgets.Output()

def run_band(b=None):
    country = w_bc.value
    prefix, label = w_bme.value, [l for l,v in w_bme.options if v==w_bme.value][0]
    months_to_use = list(reversed(ts_months[:w_bn.value]))
    with out_b:
        clear_output(wait=True)
        print(f"Loading {len(months_to_use)} months for {country} ...")
        rows = []
        for m in months_to_use:
            try:
                slice_name = "uploads_by_country" if prefix == "upload" else "downloads_by_country"
                df = load_parquet(slice_name, m)
                r = df[df["country_code"]==country]
                if r.empty: continue
                r = r.iloc[0]
                rows.append({"month": pd.to_datetime(m),
                             "p25": r.get(f"{prefix}_p25", float("nan")),
                             "p50": r.get(f"{prefix}_p50", float("nan")),
                             "p75": r.get(f"{prefix}_p75", float("nan"))})
            except Exception as e:
                print(f"  Skipping {m}: {e}")
        if not rows: print("No data loaded."); return
        band = pd.DataFrame(rows).sort_values("month")
        fig, ax = plt.subplots(figsize=(11,5))
        ax.fill_between(band["month"],band["p25"],band["p75"],alpha=0.25,label="p25-p75 band")
        ax.plot(band["month"],band["p50"],marker="o",lw=2,label="p50 (median)")
        ax.set_ylabel(label); ax.legend()
        ax.set_title(f"{cc_name(country)} — {label} over time\n"
                     "Shaded band = middle 50% of connections (p25-p75)")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(6))
        plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha="right")
        plt.tight_layout(); plt.show()

w_brun.on_click(run_band)
display(widgets.VBox([widgets.HBox([w_bc,w_bme,w_bn,w_brun]),out_b]))
run_band()